# Vanilla RAG Workshop Demo
## End-to-End Retrieval-Augmented Generation Pipeline

This notebook demonstrates a complete **Retrieval-Augmented Generation (RAG)** system built with LangChain and Groq API.

### What You'll Learn:
**Document Ingestion** - Load and parse PDF files  
**Text Chunking** - Split documents into manageable pieces  
**Embeddings** - Convert text to numerical vectors  
**Vector Storage** - Store embeddings for fast retrieval  
**Retrieval** - Find relevant content using semantic search  
**Generation** - Use LLM to answer questions based on retrieved context  

### Requirements:
- Google Colab
- GROQ_API_KEY (add to Colab Secrets)
- A PDF file to analyze

## Step 0: Install Dependencies
Run this cell first to install all required libraries. This includes LangChain, embeddings models, and vector database tools.

In [ ]:
# Install all required dependencies
!pip install -q langchain langchain-core langchain-community langchain-groq langchain-huggingface faiss-cpu pypdf google-colab python-dotenv

print("All dependencies installed successfully!")

All dependencies installed successfully!


## Step 1: Upload Your PDF Document
Upload a PDF file that you want to use for the RAG system. This file will be ingested and processed.

In [ ]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader

# Upload PDF file
print("Please upload your PDF document:")
uploaded = files.upload()

# Extract filename
file_name = list(uploaded.keys())[0]
print(f"File detected: {file_name}")

# Load PDF
loader = PyPDFLoader(file_name)
documents = loader.load()

print(f"Successfully loaded {len(documents)} pages from your PDF")
print("\nPreview of first page (first 500 characters):")
print(documents[0].page_content[:500])

Please upload your PDF document:


Saving introduction1.pdf to introduction1 (2).pdf
File detected: introduction1 (2).pdf
Successfully loaded 3 pages from your PDF

Preview of first page (first 500 characters):
In today’s digital world, data in various formats is valuable, as it is important for sources of 
information and insights. Raw data present in various forms, such as bytes, texts, multimedia, 
etc., is converted into many forms to be used by various organizations. Hence, cleaning and 
processing data is essential to derive meaningful information and insights from the data 
. 
Data wrangling is the process of converting raw and unprocessed data from one form to another 
to make it more recogniza


## Step 2: Chunk Documents
Split the PDF into smaller, manageable chunks. This helps:
- Keep context within token limits
- Improve retrieval accuracy
- Handle long documents efficiently

We use `RecursiveCharacterTextSplitter` with 800 char chunks and 150 char overlap.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Chunking Complete!")
print(f"Total chunks created: {len(chunks)}")
print(f"\nExample chunk (first 300 characters):")
print(chunks[0].page_content[:300])


Chunking Complete!
Total chunks created: 11

Example chunk (first 300 characters):
In today’s digital world, data in various formats is valuable, as it is important for sources of 
information and insights. Raw data present in various forms, such as bytes, texts, multimedia, 
etc., is converted into many forms to be used by various organizations. Hence, cleaning and 
processing da


## Step 3: Generate Embeddings
Convert text chunks into numerical vectors (embeddings). We use the `all-MiniLM-L6-v2` model from HuggingFace:
- Lightweight and efficient
- 384-dimensional vectors
- Works offline (no API calls needed)

These embeddings will be used to find semantically similar content when a query is made.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import numpy as np

# Initialize embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Test embedding - generate for the first chunk
test_embedding = embeddings.embed_query(chunks[0].page_content)

print(f"Embeddings Model Loaded!")
print(f"Model: all-MiniLM-L6-v2")
print(f"Embedding Dimension: {len(test_embedding)}")

# Show sample embedding
print(f"\nSample Embedding (First Chunk):")
print(f"First 10 values: {[round(x, 4) for x in test_embedding[:10]]}")
print(f"Last 10 values: {[round(x, 4) for x in test_embedding[-10:]]}")
print(f"Min value: {min(test_embedding):.4f}")
print(f"Max value: {max(test_embedding):.4f}")

/tmp/ipykernel_15412/3672575101.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings Model Loaded!
Model: all-MiniLM-L6-v2
Embedding Dimension: 384

Sample Embedding (First Chunk):
First 10 values: [-0.0242, 0.0648, -0.0262, 0.0267, -0.0444, -0.11, -0.0038, -0.0172, 0.0071, 0.0145]
Last 10 values: [-0.0265, 0.0105, 0.0242, 0.0608, -0.0292, -0.0262, -0.0199, 0.0008, 0.0137, -0.0553]
Min value: -0.1210
Max value: 0.1537


## Step 4: Store Embeddings in Vector Database
Create a FAISS (Facebook AI Similarity Search) vector database to store all chunk embeddings. This enables:
- Fast similarity search
- Efficient retrieval of relevant chunks
- In-memory storage (perfect for demos)

We'll store embeddings for all chunks at once.

In [ ]:
from langchain_community.vectorstores import FAISS

# Create and store embeddings in FAISS
vector_db = FAISS.from_documents(chunks, embeddings)

print(f"Vector Database Created!")
print(f"Total vectors stored: {len(chunks)}")
print(f"Database type: FAISS")
print(f"Embedding dimension: {len(test_embedding)}")

Vector Database Created!
Total vectors stored: 11
Database type: FAISS
Embedding dimension: 384


## Step 5: Load API Key from Colab Secrets
Load your API key securely from Google Colab Secrets.

**Note:** Add your GROQ_API_KEY to Colab Secrets (⚙️ > Secrets > Create new secret named "GROQ_API_KEY")

In [ ]:
from google.colab import userdata

# Load API key from Colab Secrets
groq_api_key = userdata.get("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets. Add it via Secrets")

print("GROQ API Key loaded successfully from secrets!")

GROQ API Key loaded successfully from secrets!


## Step 6: Initialize LLM (Large Language Model)
Set up Groq's LLM using their API. We use `llama-3.1-8b-instant` for:
- Fast inference
- Free API tier available
- Good quality responses

In [ ]:
from langchain_groq import ChatGroq

# Initialize Groq LLM
llm = ChatGroq(
    api_key=groq_api_key,
    model="llama-3.1-8b-instant",
    temperature=0.7
)

# Test the LLM
print("Testing LLM connection...")
test_response = llm.invoke("Say 'RAG Workshop is awesome!' in one sentence")
print(f"LLM Response: {test_response.content}")

Testing LLM connection...
LLM Response: RAG Workshop is an incredible event that truly is awesome.


## Step 7: Build the RAG Chain
Create the complete Retrieval-Augmented Generation chain that:
1. Takes a user query
2. Embeds the query
3. Retrieves relevant chunks from the vector database
4. Passes them to the LLM with a prompt
5. Returns an answer grounded in the document

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# Define the RAG prompt template
system_prompt = """
You are an AI assistant answering questions based on provided documents.

Use the following context from the document to answer the user's question:
{context}

If the answer is not in the provided context, say "I don't have enough information in the document to answer that question."
Try to be concise and cite relevant parts of the document.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# Create the combining chain (LLM + prompt)
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Create the full RAG chain (retrieval + combining)
retrieval_chain = create_retrieval_chain(
    vector_db.as_retriever(search_kwargs={"k": 3}),  # Retrieve top 3 chunks
    combine_docs_chain
)

print(f"RAG Chain created successfully!")
print(f"Retriever: FAISS Vector Store")
print(f"LLM: Groq (llama-3.1-8b-instant)")
print(f"Top-k retrieval: 3 chunks")

RAG Chain created successfully!
Retriever: FAISS Vector Store
LLM: Groq (llama-3.1-8b-instant)
Top-k retrieval: 3 chunks


## Step 8: Query the RAG System
Now we can ask questions about the uploaded document. The system will:
1. Find relevant chunks using semantic similarity
2. Pass them to the LLM
3. Generate an answer based on the document content

**Try asking questions about your document!**

In [ ]:
# Example queries - modify these to ask about your document!

# Query 1
query_1 = "What are the main topics covered in this document?"
print(f"Query 1: {query_1}")

response_1 = retrieval_chain.invoke({"input": query_1})

print(f"Retrieved context (3 chunks):")
for i, doc in enumerate(response_1.get("context", []), 1):
    print(f"\n🔹 Chunk {i}:")
    print(doc.page_content[:300] + "...\n")

print("\nLLM Answer:")
print(response_1["answer"])

Query 1: What are the main topics covered in this document?
Retrieved context (3 chunks):

🔹 Chunk 1:
In this stage, all the irregularities and inconsistencies in data are processed and 
removed.  
 
Data Transformation:  In this process, data is res tructured into a structured 
format, which may involve converting data types, renaming, arranging data, etc. 
 
Data Enrichment:  In this stage, some a...


🔹 Chunk 2:
2. Consistency 
As data wrangling structures our data in a usable format, it makes our data more 
consistent. It is very important for the business, as it helps in achieving the objectives 
and goals of the company. It is mostly used by companies that rely heavily on input 
from their users and proc...


🔹 Chunk 3:
organizing data in a desired format for better business and useful information. 
What is Data Wrangling? 
Data Wrangling is known by many names, such as Data Cleaning, Data munging, and Data 
remediation. It is the process of collecting, cleaning, and converting r

## Step 9: Try More Queries
Run multiple queries to test the RAG system. Modify the query strings below and run the cell multiple times!

In [ ]:
# Modify these queries to ask your own questions!

queries = [
    "Summarize the key findings in this document",
    "What is the most important information?",
    "Provide a detailed explanation of the main topic"
]

for i, query in enumerate(queries, 1):
    print(f"Query {i}: {query}")

    response = retrieval_chain.invoke({"input": query})

    print(f"\nAnswer:")
    print(response["answer"])

    print(f"\nSources (Top 3 relevant chunks):")
    for j, doc in enumerate(response.get("context", []), 1):
        print(f"\nSource {j}:")
        print(doc.page_content[:250] + "...")
    print("\n")

Query 1: Summarize the key findings in this document

Answer:
Based on the provided document, the key findings can be summarized as follows:

1. **Data Wrangling is Essential**: Data wrangling is a crucial process that helps collect, clean, and convert raw data into a structured format for data analysis and decision-making.

2. **Benefits of Data Wrangling**:
   - Improves consistency and accuracy of data.
   - Enhances efficiency in extracting important information.
   - Facilitates better insights and decision-making.
   - Saves time and resources.

3. **Tools Used in Data Wrangling**: Various tools are available to automate the data cleaning process and validate organizing data in a desired format.

The document highlights the significance of data wrangling in organizing data for better business insights and decision-making.

Sources (Top 3 relevant chunks):

Source 1:
2. Consistency 
As data wrangling structures our data in a usable format, it makes our data more 
consistent. It is

## RAG Architecture Summary

This is a complete **Vanilla RAG** (Retrieval-Augmented Generation) system:

```
User Question
    ↓
[Embedding] (HuggingFace: all-MiniLM-L6-v2)
    ↓
[Vector DB Search] (FAISS) → Retrieve Top 3 Chunks
    ↓
[Context + Question] → Prompt Template
    ↓
[LLM] (Groq: llama-3.1-8b-instant)
    ↓
Answer
```

### Components:
- **Document Ingestion**: PyPDF Loader
- **Text Chunking**: RecursiveCharacterTextSplitter (800 chars, 150 overlap)
- **Embeddings**: HuggingFace (384-dim vectors)
- **Vector Store**: FAISS (in-memory, fast retrieval)
- **LLM**: Groq API (llama-3.1-8b-instant)
- **Framework**: LangChain

This demo shows a production-ready RAG pipeline suitable for workshops!